In [27]:
import sys
print(sys.executable)

d:\AI Projects Data Domain\RAG_mini_project\.venv\Scripts\python.exe


## importing everything from .env file and checking APIKey is set

In [28]:
import os
from dotenv import load_dotenv
load_dotenv()

#checking API key is set for our use
if os.environ['OPENAI_API_KEY']:
    print("API key is set")

API key is set


In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)


In [31]:
response = llm.invoke("what is AI")
response.content

'AI, or artificial intelligence, is the field of making computers and software perform tasks that normally require human intelligence. This includes things like understanding language, recognizing images, learning from data, making decisions, and solving problems.\n\nKey points:\n- Narrow (weak) AI: AI that is good at a specific task (e.g., voice assistants, recommendation systems, image recognition). This is the type of AI most common today.\n- Artificial General Intelligence (AGI): a hypothetical AI that could understand, learn, and apply intelligence across a wide range of tasks as well as a human can.\n\nHow it works (high level):\n- Many AI systems learn from data using machine learning. They find patterns in large datasets to make predictions or decisions.\n- Types of learning include supervised (labeled data), unsupervised (finding structure in unlabeled data), and reinforcement learning (learning by trial and error).\n- A popular subset is deep learning, which uses large neural

# RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA - WE HAVE DATA IN NORMAL TEXT FORMAT

### step-1 Preparing document for our text

In [32]:
from langchain_core.documents import Document

#your text data
my_text = '''Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics. AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields. Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.'''

docs = [Document(page_content=my_text, metadata={'source':'ABC co.', 'documentId':'Doc1'})]
#above is used to create a document based on our text
#Our document is ready :)

docs

[Document(metadata={'source': 'ABC co.', 'documentId': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing

### strp-2 converting the document into chunks

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=50)

chunk = splitter.split_documents(docs)
chunk

[Document(metadata={'source': 'ABC co.', 'documentId': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.'),
 Document(metadata={'source': 'ABC co.', 'documentId': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.'),
 Document(metadata={'source': 'ABC co.', 'docum

# step 3: create the embeddings for these chunks.
### using openai model and using the function openAI Embeddings


In [34]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Step 4: CREATE and sTORE embeddings in vector database/vector store

In [35]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents = chunk,
    embedding = embedding_model
)

# Step 5: Sementic Search/Similarity Search

In [36]:
vectorstore.similarity_search("what are the goals of AI", k=3)

[Document(metadata={'documentId': 'Doc1', 'source': 'ABC co.'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics. AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.'),
 Document(metadata={'source': 'ABC co.', 'documentId': 'Doc1'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and 

# talk to LLM

In [38]:
context = vectorstore.similarity_search("what are the goals of AI", k=3)

response = llm.invoke(f"list me all goals of AI, you can answer using followwing context: {context}")
response.content

'From the provided context, the goals of AI include:\n\n- Learning\n- Reasoning\n- Knowledge representation\n- Planning\n- Natural language processing\n- Perception\n- Robotics (support for robotics)\n- Problem-solving\n- Decision-making\n- Goal-directed action (taking actions to maximize the chances of achieving defined goals)'

import sys
print(sys.executable)